# Ver1

In [ ]:
import Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from Bio import Entrez
import time

# PubMed APIのメールアドレス設定（ご自身のメールアドレスに変更してください）
Entrez.email = "your.email@example.com"

# 1. データの読み込み
input_file = "pubmed_inputdata.xlsx"
try:
    df_input = pd.read_excel(input_file)
    print(f"{input_file} を読み込みました。検索を開始します。\n")
except FileNotFoundError:
    print(f"エラー: {input_file} が見つかりません。作成して同じディレクトリに配置してください。")
    raise

results_summary = []
all_papers = {}

# 2. クエリの実行
for index, row in df_input.iterrows():
    target = row['Target']
    query = row['PubMed Query']

    print(f"Target名: {target} を検索中...")

    try:
        # 検索の実行 (必要に応じてretmaxを調整してください。最大10000まで可能)
        handle = Entrez.esearch(db="pubmed", term=query, retmax=500)
        record = Entrez.read(handle)
        handle.close()

        count = int(record["Count"])
        print(f"Target名: {target} - {count}件ヒット\n")

        results_summary.append({
            'Target': target,
            'PubMed Query': query,
            'Hit Count': count
        })

        if count > 0:
            id_list = record["IdList"]

            # 論文詳細情報の取得
            fetch_handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
            papers = Entrez.read(fetch_handle)
            fetch_handle.close()

            for paper in papers['PubmedArticle']:
                pmid = str(paper['MedlineCitation']['PMID'])
                article = paper['MedlineCitation']['Article']
                title = article.get('ArticleTitle', '')

                # 発行年の取得
                pub_date = article.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {})
                year = pub_date.get('Year', '')

                # ジャーナル名の取得
                journal = article.get('Journal', {}).get('Title', '')

                # アブストラクトの取得
                abstract_text = ""
                if 'Abstract' in article and 'AbstractText' in article['Abstract']:
                    abstract_list = article['Abstract']['AbstractText']
                    abstract_text = " ".join([str(abs) for abs in abstract_list])

                # 5. 重複の検知と記録
                if pmid in all_papers:
                    all_papers[pmid]['Found_in_Targets'].append(target)
                    all_papers[pmid]['Is_Duplicate'] = True
                else:
                    all_papers[pmid] = {
                        'PMID': pmid,
                        'Title': title,
                        'Year': year,
                        'Journal': journal,
                        'Abstract': abstract_text,
                        'Found_in_Targets': [target],
                        'Is_Duplicate': False
                    }

        # APIへの過負荷を防ぐためスリープ処理
        time.sleep(1)

    except Exception as e:
        print(f"エラーが発生しました（Target: {target}）: {e}")

# 6. エクセルへの出力準備
df_summary = pd.DataFrame(results_summary)

# 論文詳細リストの作成（リスト内のターゲット情報を文字列に変換）
papers_list = []
for pmid, info in all_papers.items():
    info['Found_in_Targets'] = ", ".join(info['Found_in_Targets'])
    papers_list.append(info)

df_papers = pd.DataFrame(papers_list)

# 別シートに分けてExcelへ保存
output_file = "pubmed_search_results.xlsx"
with pd.ExcelWriter(output_file) as writer:
    df_summary.to_excel(writer, sheet_name="Sheet1", index=False)
    df_papers.to_excel(writer, sheet_name="Sheet2", index=False)

print(f"すべての処理が完了しました。結果は '{output_file}' に保存されています。")

pubmed_inputdata.xlsx を読み込みました。検索を開始します。

Target名: miR-126-3p を検索中...
Target名: miR-126-3p - 0件ヒット

Target名: miR-451a を検索中...
Target名: miR-451a - 0件ヒット

Target名: miR-144-3p を検索中...
Target名: miR-144-3p - 1件ヒット

Target名: miR-20b-5p を検索中...
Target名: miR-20b-5p - 1件ヒット

Target名: miR-155-5p を検索中...
Target名: miR-155-5p - 3件ヒット

すべての処理が完了しました。結果は 'pubmed_search_results.xlsx' に保存されています。


# Ver2

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from Bio import Entrez
import time

# PubMed APIのメールアドレス設定（ご自身のメールアドレスに変更してください）
Entrez.email = "your.email@example.com"

# 1. データの読み込み
input_file = "pubmed_inputdata.xlsx"
try:
    df_input = pd.read_excel(input_file)
    print(f"{input_file} を読み込みました。検索を開始します。\n")
except FileNotFoundError:
    print(f"エラー: {input_file} が見つかりません。作成して同じディレクトリに配置してください。")
    raise

results_summary = []
papers_list = []
seen_pmids = {} # { PMID: 最初に登場した通し番号 } を記録する辞書
paper_counter = 1 # 論文の通し番号

# 2. クエリの実行
for index, row in df_input.iterrows():
    query_no = str(row.get('No', index + 1))
    target = row.get('Target', 'Unknown')
    query = row.get('PubMed Query', '')

    print(f"No.{query_no} (Target: {target}) を検索中...")

    try:
        # 検索の実行
        handle = Entrez.esearch(db="pubmed", term=query, retmax=500)
        record = Entrez.read(handle)
        handle.close()

        count = int(record["Count"])
        print(f" -> {count}件ヒット")

        # ヒット件数サマリーには常に全件記録
        results_summary.append({
            'No': query_no,
            'Target': target,
            'PubMed Query': query,
            'Hit Count': count
        })

        # --- 50件以下のクエリのみ詳細情報を取得 ---
        if 0 < count <= 50:
            id_list = record["IdList"]

            # 論文詳細情報の一括取得
            fetch_handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
            papers = Entrez.read(fetch_handle)
            fetch_handle.close()

            for paper in papers['PubmedArticle']:
                pmid = str(paper['MedlineCitation']['PMID'])
                article = paper['MedlineCitation']['Article']
                title = article.get('ArticleTitle', '')

                # 著者名の取得
                authors = []
                if 'AuthorList' in article:
                    for author in article['AuthorList']:
                        if 'LastName' in author and 'Initials' in author:
                            authors.append(f"{author['LastName']} {author['Initials']}")
                        elif 'CollectiveName' in author:
                            authors.append(author['CollectiveName'])
                author_str = ", ".join(authors)

                # 発行年とジャーナル名の取得
                pub_date = article.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {})
                year = pub_date.get('Year', '')
                journal = article.get('Journal', {}).get('Title', '')

                # 重複判定とアブストラクト処理
                abstract_text = ""
                duplicate_of = ""

                if pmid in seen_pmids:
                    # 被っている場合：アブストを空欄にし、元の通し番号を記録
                    duplicate_of = seen_pmids[pmid]
                else:
                    # 初出の場合：通し番号を記録し、英語のアブストを取得
                    seen_pmids[pmid] = paper_counter

                    if 'Abstract' in article and 'AbstractText' in article['Abstract']:
                        abstract_list = article['Abstract']['AbstractText']
                        abstract_text = " ".join([str(abs) for abs in abstract_list])

                # 論文データをリストへ追加
                papers_list.append({
                    'Paper_No': paper_counter,         # 論文ごとの通し番号
                    'Query_No': query_no,              # どのクエリでヒットしたか
                    'Target': target,                  # 検索ターゲット
                    'PMID': pmid,
                    'Title': title,
                    'Authors': author_str,
                    'Year': year,
                    'Journal': journal,
                    'Duplicate_of_No': duplicate_of,   # 被っている場合の元番号
                    'Abstract': abstract_text          # 被っている場合は空
                })

                paper_counter += 1

        elif count > 50:
            print(f"    ※50件を超えているため（{count}件）、詳細取得をスキップしました。\n")
        else:
            print("\n")

        # PubMed APIへの過負荷を防ぐためスリープ処理
        time.sleep(1)

    except Exception as e:
        print(f"エラーが発生しました（Target: {target}）: {e}")

# 3. エクセルへの出力準備
df_summary = pd.DataFrame(results_summary)
df_papers = pd.DataFrame(papers_list)

# 別シートに分けてExcelへ保存
output_file = "pubmed_search_results.xlsx"
with pd.ExcelWriter(output_file) as writer:
    df_input.to_excel(writer, sheet_name="Original_Queries", index=False)
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_papers.to_excel(writer, sheet_name="Paper_Details", index=False)

print(f"すべての処理が完了しました。結果は '{output_file}' に保存されています。")

pubmed_inputdata.xlsx を読み込みました。検索を開始します。

No.1 (Target: miRNA) を検索中...
 -> 203件ヒット
    ※50件を超えているため（203件）、詳細取得をスキップしました。

No.2 (Target: miRNA) を検索中...
 -> 2件ヒット
No.3 (Target: miRNA) を検索中...
 -> 195件ヒット
    ※50件を超えているため（195件）、詳細取得をスキップしました。

No.4 (Target: miRNA) を検索中...
 -> 13件ヒット
No.5 (Target: miRNA) を検索中...
 -> 0件ヒット


No.6 (Target: miRNA) を検索中...
 -> 114件ヒット
    ※50件を超えているため（114件）、詳細取得をスキップしました。

No.7 (Target: miRNA) を検索中...
 -> 1件ヒット
No.8 (Target: miRNA) を検索中...
 -> 109件ヒット
    ※50件を超えているため（109件）、詳細取得をスキップしました。

No.9 (Target: miRNA) を検索中...
 -> 8件ヒット
No.10 (Target: miRNA) を検索中...
 -> 0件ヒット


No.11 (Target: miRNA) を検索中...
 -> 78件ヒット
    ※50件を超えているため（78件）、詳細取得をスキップしました。

No.12 (Target: miRNA) を検索中...
 -> 1件ヒット
No.13 (Target: miRNA) を検索中...
 -> 75件ヒット
    ※50件を超えているため（75件）、詳細取得をスキップしました。

No.14 (Target: miRNA) を検索中...
 -> 5件ヒット
No.15 (Target: miRNA) を検索中...
 -> 0件ヒット


No.16 (Target: miRNA) を検索中...
 -> 31件ヒット
No.17 (Target: miRNA) を検索中...
 -> 0件ヒット


No.18 (Target: miRNA) を検索中...
 -> 3

# Ver3　最新版

In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.0 MB/s  0:00:00


In [3]:
import pandas as pd
from Bio import Entrez
import time

# PubMed APIのメールアドレス設定（ご自身のメールアドレスに変更してください）
Entrez.email = "your.email@example.com"

# 1. データの読み込み
input_file = "pubmed_inputdata.xlsx"
try:
    df_input = pd.read_excel(input_file)
    print(f"{input_file} を読み込みました。検索を開始します。\n")
except FileNotFoundError:
    print(f"エラー: {input_file} が見つかりません。作成して同じディレクトリに配置してください。")
    raise

results_summary = []
papers_list = []
md_entries = [] # 重複なしのMarkdownテキストを格納するリスト
seen_pmids = {} # { PMID: 最初に登場した通し番号 } を記録する辞書
paper_counter = 1 # 論文の通し番号

# 2. クエリの実行
for index, row in df_input.iterrows():
    query_no = str(row.get('No', index + 1))
    target = row.get('Target', 'Unknown')
    query = row.get('PubMed Query', '')

    print(f"No.{query_no} (Target: {target}) を検索中...")

    try:
        # 検索の実行
        handle = Entrez.esearch(db="pubmed", term=query, retmax=500)
        record = Entrez.read(handle)
        handle.close()

        count = int(record["Count"])
        print(f" -> {count}件ヒット")

        # ヒット件数サマリー
        results_summary.append({
            'No': query_no,
            'Target': target,
            'PubMed Query': query,
            'Hit Count': count
        })

        # --- 50件以下のクエリのみ詳細情報を取得 ---
        if 0 < count <= 50:
            id_list = record["IdList"]

            # 論文詳細情報の一括取得
            fetch_handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
            papers = Entrez.read(fetch_handle)
            fetch_handle.close()

            for paper in papers['PubmedArticle']:
                pmid = str(paper['MedlineCitation']['PMID'])
                article = paper['MedlineCitation']['Article']
                title = article.get('ArticleTitle', '')

                # 著者名の取得
                authors = []
                if 'AuthorList' in article:
                    for author in article['AuthorList']:
                        if 'LastName' in author and 'Initials' in author:
                            authors.append(f"{author['LastName']} {author['Initials']}")
                        elif 'CollectiveName' in author:
                            authors.append(author['CollectiveName'])
                author_str = ", ".join(authors)

                # 発行年とジャーナル名の取得
                pub_date = article.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {})
                year = pub_date.get('Year', '')
                journal = article.get('Journal', {}).get('Title', '')

                abstract_text = ""
                duplicate_of = ""

                # 重複判定
                if pmid in seen_pmids:
                    # 被っている場合：元の通し番号を記録（Markdownには追加しない）
                    duplicate_of = seen_pmids[pmid]
                else:
                    # 初出（ユニーク）の場合：通し番号を記録し、アブストを取得
                    seen_pmids[pmid] = paper_counter

                    if 'Abstract' in article and 'AbstractText' in article['Abstract']:
                        abstract_list = article['Abstract']['AbstractText']
                        abstract_text = " ".join([str(abs) for abs in abstract_list])

                    # Markdown用エントリーを作成（初出の論文のみ）
                    md_text = f"""## [{paper_counter}] {title}
- **PMID**: {pmid}
- **Target**: {target} (Query No: {query_no})
- **Journal**: {journal} ({year})
- **Authors**: {author_str}

### Abstract
{abstract_text if abstract_text else "*(No abstract available)*"}

---"""
                    md_entries.append(md_text)

                # Excel用リストへ追加
                papers_list.append({
                    'Paper_No': paper_counter,
                    'Query_No': query_no,
                    'Target': target,
                    'PMID': pmid,
                    'Title': title,
                    'Authors': author_str,
                    'Year': year,
                    'Journal': journal,
                    'Duplicate_of_No': duplicate_of,
                    'Abstract': abstract_text
                })

                paper_counter += 1

        elif count > 50:
            print(f"    ※50件を超えているため（{count}件）、詳細取得をスキップしました。\n")
        else:
            print("\n")

        # PubMed APIへの過負荷を防ぐためスリープ処理
        time.sleep(1)

    except Exception as e:
        print(f"エラーが発生しました（Target: {target}）: {e}")

# 3. エクセルへの出力
output_excel = "pubmed_search_results.xlsx"
df_summary = pd.DataFrame(results_summary)
df_papers = pd.DataFrame(papers_list)

with pd.ExcelWriter(output_excel) as writer:
    df_input.to_excel(writer, sheet_name="Original_Queries", index=False)
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_papers.to_excel(writer, sheet_name="Paper_Details", index=False)

print(f"Excelファイルを保存しました: '{output_excel}'")

# 4. マークダウンファイルへの出力（重複除外）
output_md = "pubmed_papers.md"
with open(output_md, "w", encoding="utf-8") as f:
    f.write("# PubMed 検索ヒット論文一覧（ユニーク）\n\n")
    f.write(f"総ユニーク論文件数: {len(md_entries)} 件\n\n")
    f.write("\n\n".join(md_entries))

print(f"Markdownファイルを保存しました: '{output_md}'")

pubmed_inputdata.xlsx を読み込みました。検索を開始します。

No.1 (Target: miR-126-3p) を検索中...
 -> 0件ヒット


No.2 (Target: miR-451a) を検索中...
 -> 0件ヒット


No.3 (Target: miR-144-3p) を検索中...
 -> 1件ヒット
No.4 (Target: miR-20b-5p) を検索中...
 -> 1件ヒット
No.5 (Target: miR-155-5p) を検索中...
 -> 4件ヒット
No.6 (Target: miR-126-3p) を検索中...
 -> 1件ヒット
No.7 (Target: miR-451a) を検索中...
 -> 1件ヒット
No.8 (Target: miR-144-3p) を検索中...
 -> 1件ヒット
No.9 (Target: miR-20b-5p) を検索中...
 -> 1件ヒット
No.10 (Target: miR-155-5p) を検索中...
 -> 8件ヒット
Excelファイルを保存しました: 'pubmed_search_results.xlsx'
Markdownファイルを保存しました: 'pubmed_papers.md'
